In [8]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [12]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [13]:
b = load_queries("sql/02b_zone_wait.sql")
display(run(b, "geo_reconcile"))
display(run(b, "zone_size_distribution").T)

,hvfhs_license_num,company,n_scoped,n_outside_nyc,n_outside_nyc_valid,n_scoped_five_boroughs,n_valid_geo,n_excluded_negative_geo
0,HV0003,Uber,15356455,529,517,15355926,15234971,120955
1,HV0005,Lyft,5049211,245,243,5048966,4970147,78819
2,ALL,All,20405666,774,760,20404892,20205118,199774


,0
n_zones,260.00
n_valid_total,20205118.00
min_n,4.00
p10_n,6987.30
p25_n,28186.50
median_n,61078.50
p75_n,114638.25
max_n,366295.00
zones_lt_100,4.00
zones_100_to_999,6.00


In [14]:
bor = run(b, "borough_wait_profile")
display(bor)
display(bor.groupby("company")[["n_scoped", "n_valid", "n_excluded_negative_wait"]].sum())

,company,pu_borough,n_zones,n_scoped,n_valid,n_excluded_negative_wait,pct_excluded,mean_min,p50_min,p75_min,p90_min,p95_min,p99_min
0,All,Staten Island,20,309217,303909,5308,1.717,6.083,5.100,7.483,10.917,13.650,20.217
1,All,Queens,69,4260487,4223481,37006,0.869,5.097,4.183,6.233,9.217,11.550,17.817
2,All,Bronx,43,2591558,2565850,25708,0.992,4.796,4.100,5.783,8.217,10.283,15.450
3,All,Brooklyn,61,5458874,5407640,51234,0.939,4.614,3.950,5.583,7.917,9.900,15.100
4,All,Manhattan,67,7784756,7704238,80518,1.034,4.165,3.533,5.117,7.250,9.050,13.867
5,Uber,Staten Island,20,241335,237645,3690,1.529,6.121,5.083,7.533,11.167,13.933,20.493
6,Uber,Queens,69,3257878,3234427,23451,0.720,5.116,4.150,6.233,9.267,11.683,18.067
7,Uber,Bronx,43,2078377,2062439,15938,0.767,4.717,4.000,5.650,8.100,10.250,15.583
8,Uber,Brooklyn,61,4068691,4039195,29496,0.725,4.581,3.883,5.517,7.917,10.017,15.433
9,Uber,Manhattan,66,5709645,5661265,48380,0.847,4.093,3.433,5.000,7.183,9.067,14.133


,n_scoped,n_valid,n_excluded_negative_wait
company,,,
All,20404892,20205118,199774
Lyft,5048966,4970147,78819
Uber,15355926,15234971,120955


In [15]:
z = run(b, "zone_wait_profile")
print(len(z), "zones | n_valid sum:", int(z["n_valid"].sum()), "| n_scoped sum:", int(z["n_scoped"].sum()))
display(z.groupby("sample_tier").agg(zones=("PULocationID", "count"), valid_rows=("n_valid", "sum")))

cols = ["pu_borough", "pu_zone", "n_valid", "pct_excluded", "pct_uber_of_valid", "p50_min", "p90_min", "p95_min"]
adequate = z[z["sample_tier"] == "3_adequate"]
print("Longest median waits (adequate zones only)")
display(adequate.sort_values("p50_min", ascending=False)[cols].head(15))
print("Shortest median waits (adequate zones only)")
display(adequate.sort_values("p50_min")[cols].head(15))
print("Longest median waits WITHOUT the sample-size rule (illustrates small-sample noise)")
display(z.sort_values("p50_min", ascending=False)[cols + ["sample_tier"]].head(10))
print("Highest excluded share (adequate zones, D12)")
display(adequate.sort_values("pct_excluded", ascending=False)[cols].head(10))

260 zones | n_valid sum: 20205118 | n_scoped sum: 20404892


,zones,valid_rows
sample_tier,,
1_very_small,4,53
2_small,6,3283
3_adequate,250,20201782


Longest median waits (adequate zones only)


,pu_borough,pu_zone,n_valid,pct_excluded,pct_uber_of_valid,p50_min,p90_min,p95_min
40,Bronx,City Island,2803,3.445,80.8,8.000,15.300,17.995
171,Queens,JFK Airport,366295,0.066,80.4,7.167,13.683,16.967
213,Queens,Hammels/Arverne,31820,2.234,79.7,6.433,14.202,17.450
254,Staten Island,Charleston/Tottenville,8598,2.173,83.4,6.183,13.783,16.633
230,Queens,Rockaway Park,11330,3.146,83.3,6.167,14.350,17.309
167,Manhattan,Randalls Island,2279,3.021,70.0,6.133,13.337,16.438
200,Queens,Far Rockaway,46519,2.160,78.8,6.117,13.450,16.617
256,Staten Island,Arden Heights,5641,3.424,79.9,5.817,11.783,14.133
253,Staten Island,Eltingville/Annadale/Prince's Bay,9163,2.334,80.7,5.733,11.967,14.732
250,Staten Island,Great Kills,13808,2.279,80.4,5.650,12.317,15.044


Shortest median waits (adequate zones only)


,pu_borough,pu_zone,n_valid,pct_excluded,pct_uber_of_valid,p50_min,p90_min,p95_min
133,Manhattan,Penn Station/Madison Sq West,123590,1.117,73.8,3.083,7.667,9.900
126,Manhattan,Flatiron,136596,0.974,73.8,3.100,6.467,7.967
118,Manhattan,Midtown North,162624,1.220,75.7,3.233,6.800,8.383
117,Manhattan,Upper East Side South,170155,0.707,77.6,3.233,6.050,7.267
197,Queens,Long Island City/Queens Plaza,48589,1.041,74.6,3.233,6.467,8.283
173,Queens,Astoria,192897,0.674,73.8,3.250,6.333,7.983
138,Manhattan,SoHo,114259,0.404,75.0,3.250,6.483,7.900
109,Manhattan,East Chelsea,214453,1.100,72.5,3.283,7.200,9.083
134,Manhattan,Greenwich Village North,120063,0.815,74.0,3.283,6.717,8.317
135,Manhattan,Sutton Place/Turtle Bay North,118141,1.117,71.6,3.283,6.117,7.450


Longest median waits WITHOUT the sample-size rule (illustrates small-sample noise)


,pu_borough,pu_zone,n_valid,pct_excluded,pct_uber_of_valid,p50_min,p90_min,p95_min,sample_tier
42,Bronx,Rikers Island,4,0.000,25.0,11.817,14.663,14.798,1_very_small
239,Queens,Jamaica Bay,26,0.000,80.8,9.675,16.533,18.763,1_very_small
236,Queens,Breezy Point/Fort Tilden/Riis Beach,579,7.063,84.8,9.650,18.510,22.495,2_small
235,Queens,Broad Channel,907,1.520,85.8,8.083,17.720,21.090,2_small
40,Bronx,City Island,2803,3.445,80.8,8.000,15.300,17.995,3_adequate
171,Queens,JFK Airport,366295,0.066,80.4,7.167,13.683,16.967,3_adequate
213,Queens,Hammels/Arverne,31820,2.234,79.7,6.433,14.202,17.450,3_adequate
258,Staten Island,Freshkills Park,437,0.228,77.8,6.267,12.240,14.103,2_small
254,Staten Island,Charleston/Tottenville,8598,2.173,83.4,6.183,13.783,16.633,3_adequate
230,Queens,Rockaway Park,11330,3.146,83.3,6.167,14.350,17.309,3_adequate


Highest excluded share (adequate zones, D12)


,pu_borough,pu_zone,n_valid,pct_excluded,pct_uber_of_valid,p50_min,p90_min,p95_min
40,Bronx,City Island,2803,3.445,80.8,8.000,15.300,17.995
256,Staten Island,Arden Heights,5641,3.424,79.9,5.817,11.783,14.133
257,Staten Island,Rossville/Woodrow,5243,3.194,83.5,5.517,11.873,14.647
230,Queens,Rockaway Park,11330,3.146,83.3,6.167,14.350,17.309
167,Manhattan,Randalls Island,2279,3.021,70.0,6.133,13.337,16.438
255,Staten Island,Oakwood,7040,2.990,79.1,5.583,11.517,14.501
165,Manhattan,Roosevelt Island,12144,2.739,76.5,5.267,10.483,12.683
229,Queens,Douglaston,12956,2.616,79.4,5.367,10.717,13.167
95,Brooklyn,Starrett City,24865,2.567,79.1,4.783,9.443,11.600
253,Staten Island,Eltingville/Annadale/Prince's Bay,9163,2.334,80.7,5.733,11.967,14.732


In [16]:
zc = run(b, "zone_wait_profile_by_company")
print(len(zc), "zone-company rows")
display(zc.groupby(["company", "sample_tier"]).agg(zones=("PULocationID", "count"), valid_rows=("n_valid", "sum")))
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
bor.to_csv(out / "2b_borough_wait_profile.csv", index=False)
z.to_csv(out / "2b_zone_wait_profile.csv", index=False)
zc.to_csv(out / "2b_zone_wait_profile_by_company.csv", index=False)

519 zone-company rows


zones  valid_rows
company sample_tier                    
Lyft    1_very_small      7         285
        2_small          13        5734
        3_adequate      240     4964128
Uber    1_very_small      3          39
        2_small           8        4237
        3_adequate      248    15230695